```
dataset/
├── hello/
│   ├── 0/   ← sequence 0
│   │   ├── 0.npy  ← frame 0 keypoints
│   │   ├── 1.npy
│   │   └── ...
│   │   └── 29.npy
│   ├── 1/
│   └── ...
├── thanks/
│   └── ...
└── iloveyou/
    └── ...
```

In [3]:
#!pip install cvzone
#!pip install mediapipe

  Using cached mediapipe-0.10.33-py3-none-manylinux_2_28_x86_64.whl.metadata (9.8 kB)
  Using cached sounddevice-0.5.5-py3-none-any.whl.metadata (1.4 kB)
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached mediapipe-0.10.33-py3-none-manylinux_2_28_x86_64.whl (11.4 MB)
Using cached sounddevice-0.5.5-py3-none-any.whl (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 MB 663.1 kB/s eta 0:00:00m eta 0:00:010:00:03
Using cached cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (219 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)


In [2]:
import cv2
import numpy as np
import os
import time
from cvzone.HandTrackingModule import HandDetector

print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"cvzone  : HandDetector imported")
print("\nAll imports successful!")

I0000 00:00:1775588877.882015  455439 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775588877.947700  455439 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775588880.044227  455439 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


OpenCV  : 4.13.0
NumPy   : 2.4.4
cvzone  : HandDetector imported

All imports successful!


In [5]:
SIGNS = ["hello", "thanks", "iloveyou"]
NUM_SEQUENCES = 30                          # Number of video clips per sign
SEQUENCE_LENGTH = 30                        # Frames per clip
DATASET_PATH = "dataset"                    # Root folder for saved data

# Feature vector size:
# Each hand has 21 landmarks × (x, y, z) = 63 features
# Two hands = 63 × 2 = 126 features
NUM_FEATURES = 126

print("=" * 45)
print("   SIGN LANGUAGE PIPELINE — CONFIG")
print("=" * 45)
print(f"  Signs         : {SIGNS}")
print(f"  Sequences/sign: {NUM_SEQUENCES}")
print(f"  Frames/seq    : {SEQUENCE_LENGTH}")
print(f"  Features/frame: {NUM_FEATURES}")
print(f"  Dataset path  : ./{DATASET_PATH}/")
print(f"  Total clips   : {len(SIGNS) * NUM_SEQUENCES}")
print("=" * 45)

   SIGN LANGUAGE PIPELINE — CONFIG
  Signs         : ['hello', 'thanks', 'iloveyou']
  Sequences/sign: 30
  Frames/seq    : 30
  Features/frame: 126
  Dataset path  : ./dataset/
  Total clips   : 90


In [6]:
def create_dataset_folders(dataset_path, signs, num_sequences):
    """
    Creates the full folder tree:
    dataset/
      sign_name/
        0/   1/   2/  ...  (NUM_SEQUENCES folders)
    """
    for sign in signs:
        for seq_idx in range(num_sequences):
            folder = os.path.join(dataset_path, sign, str(seq_idx))
            os.makedirs(folder, exist_ok=True)
            print(f"  {folder}  ← ready")

create_dataset_folders(DATASET_PATH, SIGNS, NUM_SEQUENCES)
print("\n All dataset folders created!")

  dataset/hello/0  ← ready
  dataset/hello/1  ← ready
  dataset/hello/2  ← ready
  dataset/hello/3  ← ready
  dataset/hello/4  ← ready
  dataset/hello/5  ← ready
  dataset/hello/6  ← ready
  dataset/hello/7  ← ready
  dataset/hello/8  ← ready
  dataset/hello/9  ← ready
  dataset/hello/10  ← ready
  dataset/hello/11  ← ready
  dataset/hello/12  ← ready
  dataset/hello/13  ← ready
  dataset/hello/14  ← ready
  dataset/hello/15  ← ready
  dataset/hello/16  ← ready
  dataset/hello/17  ← ready
  dataset/hello/18  ← ready
  dataset/hello/19  ← ready
  dataset/hello/20  ← ready
  dataset/hello/21  ← ready
  dataset/hello/22  ← ready
  dataset/hello/23  ← ready
  dataset/hello/24  ← ready
  dataset/hello/25  ← ready
  dataset/hello/26  ← ready
  dataset/hello/27  ← ready
  dataset/hello/28  ← ready
  dataset/hello/29  ← ready
  dataset/thanks/0  ← ready
  dataset/thanks/1  ← ready
  dataset/thanks/2  ← ready
  dataset/thanks/3  ← ready
  dataset/thanks/4  ← ready
  dataset/thanks/5  ← ready
  

In [3]:
def test_webcam(camera_index=0):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print(f"Could not open camera at index {camera_index}.")
        print("Try changing camera_index to 1 or 2 if you have multiple cameras.")
        return

    print(f"Webcam opened at index {camera_index}")
    print("Press Q inside the window to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read frame.")
            break

        frame = cv2.flip(frame, 1)   # Mirror so it feels natural
        cv2.putText(frame, "Webcam Test — Press Q to quit",
                    (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("Webcam Test", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam closed.")

test_webcam(camera_index=0)

Could not open camera at index 0.
Try changing camera_index to 1 or 2 if you have multiple cameras.


[ WARN:0@9.991] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x3a2bf980] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@9.992] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


In [9]:
def extract_keypoints(hands):

    left_hand  = np.zeros(63)   # 21 landmarks × 3 coords
    right_hand = np.zeros(63)

    for hand in hands:
        # lmList contains 21 landmarks, each is [id, x, y, z]
        lm_list = hand["lmList"]   # shape: (21, 4)
        # Flatten only x, y, z (skip id at index 0)
        coords = np.array([[lm[0], lm[1], lm[2]] for lm in lm_list]).flatten()  # (63,)

        hand_type = hand["type"]   # "Left" or "Right"
        if hand_type == "Left":
            left_hand = coords
        else:
            right_hand = coords

    return np.concatenate([left_hand, right_hand])   # (126,)


def draw_landmarks_info(frame, hands):

    for hand in hands:
        hand_type = hand["type"]
        bbox = hand["bbox"]   # x, y, w, h
        x, y, w, h = bbox

        color = (0, 255, 100) if hand_type == "Left" else (255, 100, 0)
        cv2.rectangle(frame, (x - 10, y - 10), (x + w + 10, y + h + 10), color, 2)
        cv2.putText(frame, hand_type, (x - 10, y - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # Draws each of the 21 landmarks
        for lm in hand["lmList"]:
            cx, cy = lm[0], lm[1]
            cv2.circle(frame, (cx, cy), 4, (255, 255, 255), cv2.FILLED)

    return frame


print("Helper functions defined!")
print(f"   extract_keypoints() → returns np.ndarray of shape (126,)")
print(f"   draw_landmarks_info() → draws visual feedback on frame")

Helper functions defined!
   extract_keypoints() → returns np.ndarray of shape (126,)
   draw_landmarks_info() → draws visual feedback on frame


In [13]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

def extract_keypoints(result):
    left_hand  = np.zeros(63)
    right_hand = np.zeros(63)
    if result.multi_hand_landmarks and result.multi_handedness:
        for hand_landmarks, handedness in zip(result.multi_hand_landmarks, result.multi_handedness):
            coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
            if handedness.classification[0].label == "Left":
                left_hand = coords
            else:
                right_hand = coords
    return np.concatenate([left_hand, right_hand])

print("All imports and functions ready!")

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [15]:
def preview_hand_detection(camera_index=0):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("Could not open webcam.")
        return
    print("Preview running. Press Q to quit.")
    with mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7) as hands:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)
            if result.multi_hand_landmarks:
                for hand_lms in result.multi_hand_landmarks:
                    mp_draw.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)
                kp = extract_keypoints(result)
                cv2.putText(frame, f"Hands detected | Features: {kp.shape[0]}",
                            (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,255,0), 2)
            else:
                cv2.putText(frame, "No hands detected",
                            (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,0,255), 2)
            cv2.imshow("Hand Detection Preview", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    cap.release()
    cv2.destroyAllWindows()
    print("Preview closed.")

preview_hand_detection(camera_index=0)

Preview running. Press Q to quit.


NameError: name 'mp_hands' is not defined

In [10]:
SIGNS = ["hello", "thanks", "iloveyou"]
NUM_SEQUENCES = 30
SEQUENCE_LENGTH = 30
DATASET_PATH = "dataset"

def collect_dataset(signs, num_sequences, seq_length, dataset_path, camera_index=0, countdown=3):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("Cannot open webcam.")
        return
    with mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7) as hands:
        for sign in signs:
            print(f"\nCollecting sign: '{sign}'")
            for seq_idx in range(num_sequences):
                start_time = time.time()
                while True:
                    ret, frame = cap.read()
                    if not ret: break
                    frame = cv2.flip(frame, 1)
                    elapsed = time.time() - start_time
                    remaining = max(0, countdown - int(elapsed))
                    cv2.putText(frame, f"GET READY: '{sign}' Seq {seq_idx+1}/{num_sequences}",
                                (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,0), 2)
                    cv2.putText(frame, f"Starting in {remaining}s..." if remaining > 0 else "GO!",
                                (15, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.85,
                                (0,255,0) if remaining == 0 else (0,165,255), 2)
                    cv2.imshow("Data Collection", frame)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        cap.release(); cv2.destroyAllWindows(); return
                    if elapsed >= countdown: break
                for frame_idx in range(seq_length):
                    ret, frame = cap.read()
                    if not ret: break
                    frame = cv2.flip(frame, 1)
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    result = hands.process(rgb)
                    keypoints = extract_keypoints(result)
                    if result.multi_hand_landmarks:
                        for hand_lms in result.multi_hand_landmarks:
                            mp_draw.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)
                    progress = int((frame_idx / seq_length) * frame.shape[1])
                    cv2.rectangle(frame, (0, frame.shape[0]-10), (progress, frame.shape[0]), (0,255,100), -1)
                    cv2.putText(frame, f"RECORDING '{sign}' | Seq {seq_idx+1} | Frame {frame_idx+1}/{seq_length}",
                                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,255), 2)
                    cv2.imshow("Data Collection", frame)
                    save_path = os.path.join(dataset_path, sign, str(seq_idx), f"{frame_idx}.npy")
                    np.save(save_path, keypoints)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        cap.release(); cv2.destroyAllWindows(); return
                print(f"  ✓ Seq {seq_idx:02d} saved")
    cap.release()
    cv2.destroyAllWindows()
    print("\nData collection complete!")

collect_dataset(SIGNS, NUM_SEQUENCES, SEQUENCE_LENGTH, DATASET_PATH)


📂 Collecting sign: 'hello'
  ✓ Seq 00 saved
  ✓ Seq 01 saved
  ✓ Seq 02 saved
  ✓ Seq 03 saved
  ✓ Seq 04 saved
  ✓ Seq 05 saved
  ✓ Seq 06 saved
  ✓ Seq 07 saved
  ✓ Seq 08 saved
  ✓ Seq 09 saved
  ✓ Seq 10 saved
  ✓ Seq 11 saved
  ✓ Seq 12 saved
  ✓ Seq 13 saved
  ✓ Seq 14 saved
  ✓ Seq 15 saved
  ✓ Seq 16 saved
  ✓ Seq 17 saved
  ✓ Seq 18 saved
  ✓ Seq 19 saved
  ✓ Seq 20 saved
  ✓ Seq 21 saved
  ✓ Seq 22 saved
  ✓ Seq 23 saved
  ✓ Seq 24 saved
  ✓ Seq 25 saved
  ✓ Seq 26 saved
  ✓ Seq 27 saved
  ✓ Seq 28 saved
  ✓ Seq 29 saved

📂 Collecting sign: 'thanks'
  ✓ Seq 00 saved
  ✓ Seq 01 saved
  ✓ Seq 02 saved
  ✓ Seq 03 saved
  ✓ Seq 04 saved
  ✓ Seq 05 saved
  ✓ Seq 06 saved
  ✓ Seq 07 saved
  ✓ Seq 08 saved
  ✓ Seq 09 saved
  ✓ Seq 10 saved
  ✓ Seq 11 saved
  ✓ Seq 12 saved
  ✓ Seq 13 saved
  ✓ Seq 14 saved
  ✓ Seq 15 saved
  ✓ Seq 16 saved
  ✓ Seq 17 saved
  ✓ Seq 18 saved
  ✓ Seq 19 saved
  ✓ Seq 20 saved
  ✓ Seq 21 saved
  ✓ Seq 22 saved
  ✓ Seq 23 saved
  ✓ Seq 24 saved
  ✓ Seq 

In [11]:
def verify_dataset(dataset_path, signs, num_sequences, seq_length):
    print("=" * 60)
    print("  DATASET VERIFICATION")
    print("=" * 60)
    all_ok = True

    for sign in signs:
        seq_count = 0
        frame_shape = None
        missing = []

        for seq_idx in range(num_sequences):
            for frame_idx in range(seq_length):
                fp = os.path.join(dataset_path, sign, str(seq_idx), f"{frame_idx}.npy")
                if os.path.exists(fp):
                    arr = np.load(fp)
                    frame_shape = arr.shape
                else:
                    missing.append(fp)
            seq_count += 1

        if missing:
            print(f"  {sign:<15} | {len(missing)} missing files")
            all_ok = False
        else:
            print(f"  {sign:<15} | {seq_count} seqs | {seq_length} frames | shape: {frame_shape}")

    print("=" * 60)
    if all_ok:
        print("\nDataset looks good! Ready for preprocessing.")
    else:
        print("\nSome files are missing. Re-run Cell 8 to collect them.")

verify_dataset(DATASET_PATH, SIGNS, NUM_SEQUENCES, SEQUENCE_LENGTH)

  DATASET VERIFICATION
  hello           | 30 seqs | 30 frames | shape: (126,) ✅
  thanks          | 30 seqs | 30 frames | shape: (126,) ✅
  iloveyou        | 30 seqs | 30 frames | shape: (126,) ✅

✅ Dataset looks good! Ready for preprocessing.


In [12]:
def build_dataset_arrays(dataset_path, signs, num_sequences, seq_length):
    X_list = []
    y_list = []
    label_map = {idx: sign for idx, sign in enumerate(signs)}

    for label_idx, sign in enumerate(signs):
        print(f"  Loading '{sign}' ...", end=" ")
        loaded = 0

        for seq_idx in range(num_sequences):
            sequence = []
            for frame_idx in range(seq_length):
                fp = os.path.join(dataset_path, sign, str(seq_idx), f"{frame_idx}.npy")
                if os.path.exists(fp):
                    kp = np.load(fp)
                else:
                    kp = np.zeros(126)   # Pad with zeros if file missing
                sequence.append(kp)

            X_list.append(sequence)     # (seq_length, 126)
            y_list.append(label_idx)
            loaded += 1

        print(f"{loaded} sequences loaded")

    X = np.array(X_list, dtype=np.float32)   # (samples, seq_length, 126)
    y = np.array(y_list, dtype=np.int32)      # (samples,)

    return X, y, label_map


print("Building dataset arrays...\n")
X, y, label_map = build_dataset_arrays(DATASET_PATH, SIGNS, NUM_SEQUENCES, SEQUENCE_LENGTH)

print("\n" + "=" * 45)
print("  PREPROCESSED DATA SUMMARY")
print("=" * 45)
print(f"  X shape : {X.shape}  → (samples, frames, features)")
print(f"  y shape : {y.shape}")
print(f"  dtype X : {X.dtype}")
print(f"  dtype y : {y.dtype}")
print(f"  Classes : {label_map}")
print(f"  Samples per class: {dict(zip(*np.unique(y, return_counts=True)))}")
print("=" * 45)

Building dataset arrays...

  Loading 'hello' ... 30 sequences loaded ✅
  Loading 'thanks' ... 30 sequences loaded ✅
  Loading 'iloveyou' ... 30 sequences loaded ✅

  PREPROCESSED DATA SUMMARY
  X shape : (90, 30, 126)  → (samples, frames, features)
  y shape : (90,)
  dtype X : float32
  dtype y : int32
  Classes : {0: 'hello', 1: 'thanks', 2: 'iloveyou'}
  Samples per class: {np.int32(0): np.int64(30), np.int32(1): np.int64(30), np.int32(2): np.int64(30)}
